# Transformer Encoder for NLE ML Replacement
This notebook trains a Transformer Encoder to replace a conventional DSP Nonlinear Equalizer (NLE). 
We perform delta learning (residual prediction) to correct the distorted `NLE OFF` signal towards the `NLE ON` reference.


## 1. Imports

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import copy
import math
import os


## 2. Configuration

In [ ]:
# Model Config
WINDOW = 256
STRIDE = 128
D_MODEL = 64
NHEAD = 4
NUM_LAYERS = 2
DIM_FEEDFORWARD = 128
DROPOUT = 0.1

# Training Config
BATCH_SIZE = 32
EPOCHS = 100
LR = 3e-4
VAL_SPLIT = 0.2
EARLY_STOP_PATIENCE = 15

# Loss Weights
AMP_WEIGHT = 0.5
SNAP_WEIGHT = 0.05
DC_WEIGHT = 0.1


## 3. Device and Reproducibility

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


## 4. Load Data
We load the NLE OFF (input) and NLE ON (reference) data.

In [ ]:
off_data = np.load('data/nle_off_10k.npy')
on_data = np.load('data/nle_on_10k.npy')

print("NLE OFF Shape:", off_data.shape)
print("NLE ON Shape:", on_data.shape)

# Normalize the data (using standard scaling to avoid very small gradients)
# We will fit the scaler on the whole dataset for simplicity here, but ideally on train set only.
# Given the communication signal properties, zero mean and variance 1 scaling is appropriate.
mean_val = np.mean(off_data, axis=0)
std_val = np.std(off_data, axis=0)

off_data_norm = (off_data - mean_val) / std_val
on_data_norm = (on_data - mean_val) / std_val

print("Normalized data mean:", np.mean(off_data_norm, axis=0), "std:", np.std(off_data_norm, axis=0))


## 5. Clean Train/Validation Split
To avoid data leakage, we split the continuous signal first before creating overlapping windows.

In [ ]:
total_samples = len(off_data_norm)
train_size = int(total_samples * (1 - VAL_SPLIT))

train_off = off_data_norm[:train_size]
train_on = on_data_norm[:train_size]

val_off = off_data_norm[train_size:]
val_on = on_data_norm[train_size:]

print("Train split size:", len(train_off))
print("Val split size:", len(val_off))


## 6. Sequence Window Dataset
Generate overlapping sequence windows of size 256 with a stride of 128.

In [ ]:
class NLEWindowDataset(Dataset):
    def __init__(self, x_data, y_data, window, stride):
        self.x_windows = []
        self.delta_windows = []
        
        # Calculate delta to learn
        delta_data = y_data - x_data
        
        num_windows = (len(x_data) - window) // stride + 1
        for i in range(num_windows):
            start = i * stride
            end = start + window
            self.x_windows.append(x_data[start:end])
            self.delta_windows.append(delta_data[start:end])
            
    def __len__(self):
        return len(self.x_windows)
        
    def __getitem__(self, idx):
        return torch.tensor(self.x_windows[idx], dtype=torch.float32),                torch.tensor(self.delta_windows[idx], dtype=torch.float32)

train_dataset = NLEWindowDataset(train_off, train_on, WINDOW, STRIDE)
val_dataset = NLEWindowDataset(val_off, val_on, WINDOW, STRIDE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")


## 7. Transformer Model and Positional Encoding
Positional encoding is necessary because self-attention is permutation invariant. Without it, the model wouldn't know the temporal order of the samples.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        return x + self.pe[:, :x.size(1), :]

class TransformerNLE(nn.Module):
    def __init__(self, d_model, nhead, num_layers, dim_feedforward, dropout):
        super().__init__()
        self.input_linear = nn.Linear(2, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, 
            dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        
        self.output_linear = nn.Linear(d_model, 2)
        
    def forward(self, src):
        # src: (batch_size, seq_len, 2)
        x = self.input_linear(src)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        out = self.output_linear(x)
        return out

model = TransformerNLE(D_MODEL, NHEAD, NUM_LAYERS, DIM_FEEDFORWARD, DROPOUT).to(device)
print(model)


## 8. Domain-Aware Loss Function
We include MSE, amplitude preservation, QPSK snapping, and DC-centering.

In [ ]:
def domain_aware_loss(pred_delta, true_delta, x):
    # 1. IQ Correction MSE
    mse_iq = torch.mean((pred_delta - true_delta)**2)
    
    # 2. Amplitude Preservation
    y_hat = x + pred_delta
    y_true = x + true_delta
    
    amp_hat = torch.sqrt(y_hat[..., 0]**2 + y_hat[..., 1]**2 + 1e-8)
    amp_true = torch.sqrt(y_true[..., 0]**2 + y_true[..., 1]**2 + 1e-8)
    mse_amp = torch.mean((amp_hat - amp_true)**2)
    
    # 3. QPSK Snapping (normalized by scaler, so ideal points change slightly, but we use the concept)
    # The actual ideal points depend on the original signal scaling. 
    # For a zero-mean unit-variance signal, we approximate ideal radius as 1.
    ideal_mag = 1.0
    ideal_points = torch.tensor([
        [-0.707, -0.707], [-0.707, 0.707],
        [0.707, -0.707], [0.707, 0.707]
    ], device=x.device) * ideal_mag
    
    # Distance to closest point for each sample in y_hat
    # y_hat shape: (B, S, 2). ideal_points shape: (4, 2)
    # y_hat_expanded: (B, S, 4, 2)
    distances = torch.sum((y_hat.unsqueeze(2) - ideal_points.view(1, 1, 4, 2))**2, dim=-1)
    min_distances, _ = torch.min(distances, dim=-1)
    snap_loss = torch.mean(min_distances)
    
    # 4. DC Centering
    # Penalize centroid displacement from (0,0)
    mean_I = torch.mean(y_hat[..., 0])
    mean_Q = torch.mean(y_hat[..., 1])
    dc_loss = mean_I**2 + mean_Q**2
    
    total_loss = mse_iq + AMP_WEIGHT * mse_amp + SNAP_WEIGHT * snap_loss + DC_WEIGHT * dc_loss
    return total_loss, mse_iq.item()


## 9. Optimizer and Scheduler

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


## 10. Training Loop

In [ ]:
train_losses = []
val_losses = []
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    epoch_train_loss = 0.0
    for x_batch, delta_batch in train_loader:
        x_batch, delta_batch = x_batch.to(device), delta_batch.to(device)
        
        optimizer.zero_grad()
        pred_delta = model(x_batch)
        loss, _ = domain_aware_loss(pred_delta, delta_batch, x_batch)
        
        loss.backward()
        optimizer.step()
        epoch_train_loss += loss.item()
        
    scheduler.step()
    epoch_train_loss /= len(train_loader)
    train_losses.append(epoch_train_loss)
    
    # Validation
    model.eval()
    epoch_val_loss = 0.0
    with torch.no_grad():
        for x_batch, delta_batch in val_loader:
            x_batch, delta_batch = x_batch.to(device), delta_batch.to(device)
            pred_delta = model(x_batch)
            loss, _ = domain_aware_loss(pred_delta, delta_batch, x_batch)
            epoch_val_loss += loss.item()
            
    epoch_val_loss /= len(val_loader)
    val_losses.append(epoch_val_loss)
    
    print(f"Epoch {epoch+1:03d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")
    
    # Early Stopping
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), 'transformer_nle_best.pth')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print("Early stopping triggered!")
            break


## 11. Loss Curves

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.show()


## 12. Inference and Overlapping Window Reconstruction
Load the best model and predict on the entire dataset. Accumulate overlapping predictions and average them.

In [ ]:
model.load_state_dict(torch.load('transformer_nle_best.pth', map_location=device, weights_only=True))
model.eval()

# Reconstruct entire off_data_norm sequence
full_dataset = NLEWindowDataset(off_data_norm, on_data_norm, WINDOW, STRIDE)
full_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=False)

accumulated_preds = np.zeros_like(off_data_norm)
overlap_counts = np.zeros(len(off_data_norm))

idx = 0
with torch.no_grad():
    for x_batch, _ in full_loader:
        x_batch = x_batch.to(device)
        pred_delta_batch = model(x_batch).cpu().numpy()
        
        for i in range(len(pred_delta_batch)):
            start = (idx + i) * STRIDE
            end = start + WINDOW
            
            accumulated_preds[start:end] += pred_delta_batch[i]
            overlap_counts[start:end] += 1
            
        idx += len(pred_delta_batch)

# Avoid divide by zero
overlap_counts[overlap_counts == 0] = 1
final_delta_hat = accumulated_preds / overlap_counts[:, None]
final_y_hat_norm = off_data_norm + final_delta_hat

# Inverse transform normalization
final_y_hat = final_y_hat_norm * std_val + mean_val


## 13. Constellation Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

def plot_constellation(ax, data, title):
    ax.scatter(data[:, 0], data[:, 1], s=1, alpha=0.5, color='b')
    ax.set_title(title)
    ax.set_xlabel('I')
    ax.set_ylabel('Q')
    ax.grid(True)
    ax.set_xlim([-3, 3])
    ax.set_ylim([-3, 3])
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)

plot_constellation(axes[0], off_data, "DSP NLE OFF (Distorted)")
plot_constellation(axes[1], final_y_hat, "Transformer ML NLE")
plot_constellation(axes[2], on_data, "DSP NLE ON (Reference)")

plt.tight_layout()
plt.show()


## 14. EVM Calculation
Error Vector Magnitude (EVM) calculation.

In [ ]:
def calculate_evm(signal, reference):
    error = signal - reference
    evm_rms = np.sqrt(np.mean(np.sum(error**2, axis=1))) / np.sqrt(np.mean(np.sum(reference**2, axis=1)))
    return evm_rms * 100

evm_off = calculate_evm(off_data, on_data)
evm_ml = calculate_evm(final_y_hat, on_data)

print(f"EVM (DSP NLE OFF): {evm_off:.2f}%")
print(f"EVM (Transformer ML NLE): {evm_ml:.2f}%")


## 15. BER Calculation
Standard QPSK mapping to calculate BER.

In [ ]:
def qpsk_demodulate(signal):
    bits = np.zeros((len(signal), 2), dtype=int)
    bits[:, 0] = (signal[:, 0] > 0).astype(int)
    bits[:, 1] = (signal[:, 1] > 0).astype(int)
    return bits

bits_ref = qpsk_demodulate(on_data)
bits_off = qpsk_demodulate(off_data)
bits_ml = qpsk_demodulate(final_y_hat)

ber_off = np.mean(bits_off != bits_ref)
ber_ml = np.mean(bits_ml != bits_ref)

print(f"BER (DSP NLE OFF) vs Reference: {ber_off:.6f}")
print(f"BER (Transformer ML NLE) vs Reference: {ber_ml:.6f}")


## 16. Conclusion
The notebook demonstrates the performance of the Transformer model in predicting the NLE correction. The metrics and constellation plots above will dictate whether Transformer > MLP, Transformer ≈ MLP, or Transformer < MLP.